# 🎙️ Garhwali TTS Training — Ghughuti AI

Fine-tune a **VITS** model on Garhwali audio for Ghughuti AI voice synthesis.

**Two data sources:**
- **Community recordings** from pahaditube.in/voice-recording (clean, transcribed)
- **YouTube Garhwali audio** — auto-downloaded, segmented, and transcribed with Whisper

**Pipeline:**
1. Install dependencies
2. Download community recordings from PahadiTube API
3. *(Bonus)* Download + segment + transcribe YouTube Garhwali audio
4. Merge all data → verify dataset
5. Fine-tune VITS from base model
6. Test & export → deploy to HuggingFace

**Requirements:**
- Google Colab **GPU runtime** (T4 free tier works)
- Minimum **50 samples** recommended (community + YouTube combined)

## 1️⃣ Setup & Install Dependencies

In [ ]:
# Check GPU availability
!nvidia-smi

# Install Coqui TTS + dependencies
!pip install -q TTS==0.22.0 librosa soundfile pydub requests tqdm

# ffmpeg for audio conversion
!apt-get -qq install ffmpeg

print("\n✅ All dependencies installed!")

In [ ]:
# Verify GPU and imports
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 2️⃣ Download Recordings from PahadiTube

In [ ]:
import os, requests, json
from tqdm import tqdm

# Create dataset directories
os.makedirs('/content/garhwali_dataset/wavs', exist_ok=True)

# ── Configuration ──
# Replace with your actual deployed URL
API_URL = os.environ.get("PAHADITUBE_API", "https://pahaditube.in/api/tts")
ADMIN_KEY = os.environ.get("ADMIN_KEY", "")  # Set in Colab secrets if needed

headers = {}
if ADMIN_KEY:
    headers["X-Admin-Key"] = ADMIN_KEY

# Fetch recordings metadata
print(f"Fetching recordings from {API_URL}...")
try:
    response = requests.get(f"{API_URL}/recordings", headers=headers, timeout=30)
    response.raise_for_status()
    data = response.json()
    recordings = data.get('recordings', [])
    print(f"✅ Found {len(recordings)} recordings")

    if len(recordings) < 50:
        print(f"\n⚠️  Only {len(recordings)} recordings available.")
        print("  For decent quality, aim for 50+. For good quality, 100+.")
        print("  Ask people to record at: pahaditube.in/voice-recording")
except Exception as e:
    print(f"❌ Error fetching recordings: {e}")
    print("\nOptions:")
    print("  1. Check if your server is running and API URL is correct")
    print("  2. Upload a ZIP manually (run the next cell)")
    recordings = []

print(f"\nRecordings to process: {len(recordings)}")

In [ ]:
# Download and convert audio files
from pydub import AudioSegment
import soundfile as sf
import numpy as np

metadata_lines = []
downloaded = 0
skipped = 0

for rec in tqdm(recordings, desc="Downloading"):
    sid = rec.get('sentenceId', rec.get('id', f'unk_{downloaded}'))
    text = rec.get('text', '').strip()
    audio_url = rec.get('audio_url', rec.get('url', ''))

    if not text or not audio_url:
        skipped += 1
        continue

    try:
        # Download audio
        audio_response = requests.get(audio_url, timeout=30)
        audio_response.raise_for_status()

        # Detect format from content-type or URL
        ext = 'webm'
        ct = audio_response.headers.get('content-type', '')
        if 'wav' in ct or audio_url.endswith('.wav'):
            ext = 'wav'
        elif 'mp3' in ct or audio_url.endswith('.mp3'):
            ext = 'mp3'
        elif 'ogg' in ct or audio_url.endswith('.ogg'):
            ext = 'ogg'

        temp_path = f"/content/temp_{sid}.{ext}"
        with open(temp_path, 'wb') as f:
            f.write(audio_response.content)

        # Convert to WAV (22050 Hz, mono, 16-bit)
        audio = AudioSegment.from_file(temp_path)
        audio = audio.set_frame_rate(22050).set_channels(1).set_sample_width(2)

        # Skip if too short (<0.5s) or too long (>30s)
        duration_s = len(audio) / 1000
        if duration_s < 0.5 or duration_s > 30:
            skipped += 1
            os.remove(temp_path)
            continue

        wav_path = f"/content/garhwali_dataset/wavs/{sid}.wav"
        audio.export(wav_path, format='wav')

        # LJSpeech format: filename|text (no extension in filename)
        metadata_lines.append(f"{sid}|{text}")
        downloaded += 1

        os.remove(temp_path)

    except Exception as e:
        print(f"  ⚠️ Skipped {sid}: {e}")
        skipped += 1

# Save metadata.csv
with open('/content/garhwali_dataset/metadata.csv', 'w', encoding='utf-8') as f:
    f.write('\n'.join(metadata_lines))

print(f"\n✅ Downloaded: {downloaded} recordings")
print(f"⏭️  Skipped: {skipped}")
print(f"📄 Metadata: /content/garhwali_dataset/metadata.csv")

### Alternative: Upload ZIP manually

If API download doesn't work, upload a ZIP file with:
```
garhwali_dataset/
├── wavs/
│   ├── garhwali_001.wav
│   └── ...
└── metadata.csv
```

In [ ]:
# Upload ZIP manually (run this cell, then use the upload button)
from google.colab import files

print("Upload your dataset ZIP file:")
uploaded = files.upload()

for filename in uploaded.keys():
    if filename.endswith('.zip'):
        !unzip -o "{filename}" -d /content/
        print(f"✅ Extracted {filename}")

## 2b️⃣ Bonus: Download Garhwali Audio from YouTube

Use YouTube Garhwali speech/songs as **additional training data**.
- Downloads audio from YouTube videos via yt-dlp
- Segments into sentence-length clips using Voice Activity Detection
- Auto-transcribes with Whisper (Hindi mode — closest to Garhwali)
- Merges with community recordings

⚠️ **Use only for personal/research purposes. Respect copyright.**

In [ ]:
# Install YouTube + Whisper dependencies
!pip install -q yt-dlp openai-whisper silero-vad torch torchaudio

print("✅ YouTube pipeline dependencies installed")

In [ ]:
# ── YouTube Video URLs ──
# Add Garhwali speech videos here (interviews, stories, podcasts work best)
# Avoid songs with heavy music — clean speech gives better TTS results

YOUTUBE_URLS = [
    # Garhwali interviews / storytelling / podcasts
    # Add your URLs below — example format:
    # "https://www.youtube.com/watch?v=XXXXXXXXXXX",
]

# Or search and download automatically
YOUTUBE_SEARCH_QUERIES = [
    "Garhwali language lesson",
    "Garhwali folk story telling",
    "Garhwali interview podcast",
    "गढ़वाली भाषा बोलचाल",
    "गढ़वाली कहानी सुनाओ",
]

MAX_VIDEOS_PER_QUERY = 3   # Videos to download per search query
MAX_DURATION_MIN = 15       # Skip videos longer than this (avoid full movies)

print(f"Direct URLs: {len(YOUTUBE_URLS)}")
print(f"Search queries: {len(YOUTUBE_SEARCH_QUERIES)} (×{MAX_VIDEOS_PER_QUERY} each)")
print(f"Max video duration: {MAX_DURATION_MIN} min")

In [ ]:
# ── Step 1: Download audio from YouTube ──
import subprocess, glob, os

yt_audio_dir = '/content/yt_audio_raw'
os.makedirs(yt_audio_dir, exist_ok=True)

def download_yt_audio(url, output_dir):
    """Download audio from a single YouTube URL as WAV."""
    try:
        cmd = [
            'yt-dlp',
            '--extract-audio',
            '--audio-format', 'wav',
            '--audio-quality', '0',
            '--max-downloads', '1',
            '--match-filter', f'duration<{MAX_DURATION_MIN * 60}',
            '--no-playlist',
            '--output', f'{output_dir}/%(id)s.%(ext)s',
            '--quiet',
            url
        ]
        subprocess.run(cmd, check=True, timeout=120)
        return True
    except Exception as e:
        print(f"  ⚠️ Failed: {e}")
        return False

# Download from direct URLs
print("📥 Downloading from direct URLs...")
for url in YOUTUBE_URLS:
    print(f"  → {url}")
    download_yt_audio(url, yt_audio_dir)

# Download from search queries
print(f"\n🔍 Searching YouTube...")
for query in YOUTUBE_SEARCH_QUERIES:
    print(f"  Searching: {query}")
    try:
        cmd = [
            'yt-dlp',
            '--extract-audio',
            '--audio-format', 'wav',
            '--audio-quality', '0',
            f'--max-downloads={MAX_VIDEOS_PER_QUERY}',
            '--match-filter', f'duration<{MAX_DURATION_MIN * 60}',
            '--no-playlist',
            '--output', f'{yt_audio_dir}/%(id)s.%(ext)s',
            '--quiet',
            f'ytsearch{MAX_VIDEOS_PER_QUERY}:{query}'
        ]
        subprocess.run(cmd, check=True, timeout=300)
    except Exception as e:
        print(f"    ⚠️ {e}")

wav_count = len(glob.glob(f'{yt_audio_dir}/*.wav'))
print(f"\n✅ Downloaded {wav_count} audio files")

In [ ]:
# ── Step 2: Segment audio into sentence-length clips using VAD ──
import torch, torchaudio, glob, os
from pydub import AudioSegment

SAMPLE_RATE = 16000  # Silero VAD requires 16kHz
MIN_CLIP_SEC = 1.0
MAX_CLIP_SEC = 15.0
SILENCE_PAD_MS = 200  # Padding around speech segments

# Load Silero VAD
print("Loading Voice Activity Detector...")
vad_model, utils = torch.hub.load('snakers4/silero-vad', 'silero_vad', force_reload=False)
(get_speech_timestamps, _, read_audio, _, _) = utils
print("✅ VAD loaded")

yt_clips_dir = '/content/yt_clips'
os.makedirs(yt_clips_dir, exist_ok=True)

clip_count = 0
raw_files = sorted(glob.glob(f'{yt_audio_dir}/*.wav'))
print(f"\nSegmenting {len(raw_files)} audio files...\n")

for audio_path in raw_files:
    vid_id = os.path.splitext(os.path.basename(audio_path))[0]
    try:
        # Load audio for VAD (16kHz)
        wav = read_audio(audio_path, sampling_rate=SAMPLE_RATE)

        # Detect speech segments
        timestamps = get_speech_timestamps(wav, vad_model, sampling_rate=SAMPLE_RATE)

        if not timestamps:
            print(f"  {vid_id}: no speech detected, skipping")
            continue

        # Load full audio in pydub for slicing (original quality)
        full_audio = AudioSegment.from_wav(audio_path)

        # Merge nearby segments and extract clips
        merged = []
        current = timestamps[0]
        for ts in timestamps[1:]:
            gap_ms = (ts['start'] - current['end']) / SAMPLE_RATE * 1000
            merged_duration = (ts['end'] - current['start']) / SAMPLE_RATE
            if gap_ms < 500 and merged_duration < MAX_CLIP_SEC:
                current['end'] = ts['end']
            else:
                merged.append(current)
                current = ts
        merged.append(current)

        extracted = 0
        for seg in merged:
            start_ms = max(0, int(seg['start'] / SAMPLE_RATE * 1000) - SILENCE_PAD_MS)
            end_ms = int(seg['end'] / SAMPLE_RATE * 1000) + SILENCE_PAD_MS
            duration_s = (end_ms - start_ms) / 1000

            if duration_s < MIN_CLIP_SEC or duration_s > MAX_CLIP_SEC:
                continue

            clip = full_audio[start_ms:end_ms]
            # Convert to 22050 Hz mono 16-bit for TTS
            clip = clip.set_frame_rate(22050).set_channels(1).set_sample_width(2)

            clip_name = f"yt_{vid_id}_{clip_count:04d}"
            clip.export(f"{yt_clips_dir}/{clip_name}.wav", format='wav')
            clip_count += 1
            extracted += 1

        print(f"  {vid_id}: {extracted} clips extracted")

    except Exception as e:
        print(f"  {vid_id}: error — {e}")

print(f"\n✅ Total clips extracted: {clip_count}")

In [ ]:
# ── Step 3: Auto-transcribe clips with Whisper ──
import whisper, glob, os
from tqdm import tqdm

print("Loading Whisper model (medium — best balance for Hindi/Garhwali)...")
print("This downloads ~1.5 GB on first run\n")
model = whisper.load_model("medium")
print("✅ Whisper loaded\n")

yt_clips = sorted(glob.glob(f'{yt_clips_dir}/*.wav'))
print(f"Transcribing {len(yt_clips)} clips...\n")

yt_metadata = []  # (filename_without_ext, transcription)
skipped = 0

for clip_path in tqdm(yt_clips, desc="Transcribing"):
    clip_name = os.path.splitext(os.path.basename(clip_path))[0]
    try:
        result = model.transcribe(
            clip_path,
            language="hi",          # Hindi — closest to Garhwali in Whisper
            task="transcribe",
            fp16=torch.cuda.is_available(),
        )
        text = result['text'].strip()

        # Quality filters
        if not text or len(text) < 5:
            skipped += 1
            continue
        # Skip if mostly English/numbers (likely not Garhwali speech)
        devanagari_chars = sum(1 for c in text if '\u0900' <= c <= '\u097F')
        if devanagari_chars < len(text) * 0.3:
            skipped += 1
            continue
        # Skip very repetitive text (music/noise artifacts)
        words = text.split()
        if len(words) > 3 and len(set(words)) < len(words) * 0.3:
            skipped += 1
            continue

        yt_metadata.append((clip_name, text))

    except Exception as e:
        skipped += 1

print(f"\n✅ Transcribed: {len(yt_metadata)} clips")
print(f"⏭️  Skipped (low quality/non-Garhwali): {skipped}")

# Preview samples
print("\n📝 Sample transcriptions:")
for name, text in yt_metadata[:10]:
    print(f"  {name}: {text}")

In [ ]:
# ── Step 4: Merge YouTube clips into main dataset ──
import shutil, os

dataset_dir = '/content/garhwali_dataset'
wav_dir = f'{dataset_dir}/wavs'
metadata_file = f'{dataset_dir}/metadata.csv'

# Read existing metadata (from community recordings)
existing_lines = []
if os.path.exists(metadata_file):
    with open(metadata_file, 'r', encoding='utf-8') as f:
        existing_lines = [l.strip() for l in f if l.strip()]

print(f"Existing community recordings: {len(existing_lines)}")
print(f"YouTube clips to add: {len(yt_metadata)}")

# Copy YouTube clips to dataset and append metadata
added = 0
for clip_name, text in yt_metadata:
    src = f"{yt_clips_dir}/{clip_name}.wav"
    dst = f"{wav_dir}/{clip_name}.wav"
    if os.path.exists(src):
        shutil.copy2(src, dst)
        existing_lines.append(f"{clip_name}|{text}")
        added += 1

# Write merged metadata
with open(metadata_file, 'w', encoding='utf-8') as f:
    f.write('\n'.join(existing_lines))

total = len(existing_lines)
print(f"\n✅ Added {added} YouTube clips to dataset")
print(f"📊 Total dataset: {total} samples")

if total >= 200:
    print("🟢 Great dataset size!")
elif total >= 50:
    print("🟡 Decent size — more data always helps")
else:
    print("🔴 Small dataset — model quality will be limited")

## 3️⃣ Prepare Dataset & Config

In [ ]:
# Verify dataset
import os, librosa, numpy as np

wav_dir = '/content/garhwali_dataset/wavs'
metadata_file = '/content/garhwali_dataset/metadata.csv'

wav_files = [f for f in os.listdir(wav_dir) if f.endswith('.wav')]
print(f"WAV files: {len(wav_files)}")

with open(metadata_file, 'r', encoding='utf-8') as f:
    lines = [l.strip() for l in f if l.strip()]
print(f"Metadata entries: {len(lines)}")

# Audio stats
durations = []
for wav_file in wav_files:
    try:
        y, sr = librosa.load(os.path.join(wav_dir, wav_file), sr=None)
        durations.append(len(y) / sr)
    except:
        pass

total_min = sum(durations) / 60
avg_sec = np.mean(durations) if durations else 0

print(f"\n📊 Audio Statistics:")
print(f"  Total duration: {total_min:.1f} minutes")
print(f"  Average clip: {avg_sec:.1f} seconds")
print(f"  Shortest: {min(durations):.1f}s | Longest: {max(durations):.1f}s")

if total_min < 10:
    print("\n🔴 <10 min — model will be very basic. Record more!")
elif total_min < 30:
    print("\n🟡 10-30 min — decent for a first model, more data helps")
else:
    print("\n🟢 30+ min — good dataset size for fine-tuning")

print("\nSample metadata:")
for line in lines[:5]:
    print(f"  {line}")

## 4️⃣ Fine-tune VITS Model

In [ ]:
# Create training config
from TTS.tts.configs.vits_config import VitsConfig

# Adjust batch size based on dataset size
num_samples = len(wav_files)
batch_size = 8 if num_samples < 100 else 16
epochs = 1000 if num_samples < 100 else 500

print(f"Dataset: {num_samples} samples → batch_size={batch_size}, epochs={epochs}")

config = VitsConfig(
    run_name="garhwali-vits",
    run_description="Garhwali TTS fine-tuned for Ghughuti AI",

    # Audio settings (22050 Hz standard for VITS)
    audio={
        "sample_rate": 22050,
        "win_length": 1024,
        "hop_length": 256,
        "num_mels": 80,
        "mel_fmin": 0,
        "mel_fmax": None,
    },

    # Training
    batch_size=batch_size,
    eval_batch_size=4,
    num_loader_workers=2,
    num_eval_loader_workers=1,
    epochs=epochs,
    save_step=500,
    save_checkpoints=True,
    save_best_after=1000,
    print_step=50,
    mixed_precision=True,  # Faster on T4

    # Garhwali uses Devanagari — no phoneme conversion needed
    use_phonemes=False,
    text_cleaner="basic_cleaners",

    # Learning rate — lower for fine-tuning
    lr_gen=0.0001,
    lr_disc=0.0001,

    # Dataset
    datasets=[{
        "formatter": "ljspeech",
        "meta_file_train": "metadata.csv",
        "path": "/content/garhwali_dataset/",
        "language": "hi",
    }],

    output_path="/content/garhwali_output/",
)

config.save_json("/content/garhwali_config.json")
print("✅ Config saved")

In [ ]:
# Download pre-trained base model
# Using LJSpeech VITS as base (best documented for fine-tuning)
# Alternative: use a Hindi model if available on Coqui hub
from TTS.api import TTS
import os

print("Downloading base VITS model for fine-tuning...")
print("(This takes 2-5 minutes on first run)\n")

tts = TTS("tts_models/en/ljspeech/vits", progress_bar=True)

# Locate the downloaded model
model_dir = os.path.expanduser("~/.local/share/tts/tts_models--en--ljspeech--vits")
model_file = os.path.join(model_dir, "model_file.pth")

if os.path.exists(model_file):
    print(f"\n✅ Base model ready: {model_file}")
    print(f"   Size: {os.path.getsize(model_file) / 1e6:.0f} MB")
else:
    # Try alternate location
    for root, dirs, files in os.walk(os.path.expanduser("~/.local/share/tts")):
        for f in files:
            if f.endswith('.pth'):
                model_file = os.path.join(root, f)
                model_dir = root
                print(f"✅ Found model at: {model_file}")
                break

In [ ]:
# ── Start fine-tuning ──
# This will take 1-4 hours on T4 depending on dataset size and epochs
# You can monitor loss in the output — it should decrease over time
# Training auto-saves checkpoints every 500 steps

import time
start = time.time()

!python -m TTS.bin.train_tts \
    --config_path /content/garhwali_config.json \
    --restore_path {model_file}

elapsed = (time.time() - start) / 60
print(f"\n🏁 Training completed in {elapsed:.0f} minutes")

## 5️⃣ Test the Model

In [ ]:
# Find the best checkpoint
import os, glob

output_dir = "/content/garhwali_output/"

# Look for best_model first, then latest checkpoint
best = glob.glob(os.path.join(output_dir, "**", "best_model*.pth"), recursive=True)
if best:
    best_model = sorted(best)[-1]
    print(f"✅ Best model: {best_model}")
else:
    all_ckpt = sorted(glob.glob(os.path.join(output_dir, "**", "*.pth"), recursive=True))
    if all_ckpt:
        best_model = all_ckpt[-1]
        print(f"⚠️ No best_model found, using latest checkpoint: {best_model}")
    else:
        print("❌ No checkpoints found! Training may have failed.")
        best_model = None

# Find config
config_candidates = glob.glob(os.path.join(output_dir, "**", "config.json"), recursive=True)
train_config = config_candidates[0] if config_candidates else "/content/garhwali_config.json"
print(f"📄 Config: {train_config}")

In [ ]:
# Test synthesis with Garhwali sentences
from TTS.api import TTS
from IPython.display import Audio, display

if best_model:
    print("Loading fine-tuned model...")
    tts = TTS(model_path=best_model, config_path=train_config, progress_bar=False)

    test_sentences = [
        "नमस्कार, तुम कैसा छा?",
        "गढ़वाल बड़ो सुंदर छ",
        "मी घर जान्दू",
        "जय बदरी विशाल",
        "आज मौसम बड़ो अच्छो छ",
        "खाणा बड़ो स्वादिष्ट छ",
    ]

    print("\n🔊 Test Results:\n")
    for i, text in enumerate(test_sentences):
        out_path = f"/content/test_{i+1}.wav"
        tts.tts_to_file(text=text, file_path=out_path)
        print(f"{i+1}. {text}")
        display(Audio(out_path, autoplay=False))
        print()
else:
    print("No model to test — check training output above.")

## 6️⃣ Export Model for Deployment

In [ ]:
# Package model for deployment to HuggingFace
import shutil, json

export_dir = "/content/garhwali_tts_export"
os.makedirs(export_dir, exist_ok=True)

if best_model:
    shutil.copy(best_model, os.path.join(export_dir, "model.pth"))
    shutil.copy(train_config, os.path.join(export_dir, "config.json"))

    info = {
        "name": "Garhwali VITS TTS — Ghughuti AI",
        "language": "Garhwali (गढ़वाली)",
        "script": "Devanagari",
        "base_model": "tts_models/en/ljspeech/vits",
        "training_samples": len(wav_files),
        "total_audio_minutes": round(total_min, 1),
        "trained_epochs": epochs,
        "sample_rate": 22050,
    }
    with open(os.path.join(export_dir, "info.json"), "w") as f:
        json.dump(info, f, indent=2, ensure_ascii=False)

    shutil.make_archive("/content/garhwali_tts_model", "zip", export_dir)
    print("✅ Model exported to /content/garhwali_tts_model.zip")
    print(f"   Size: {os.path.getsize('/content/garhwali_tts_model.zip') / 1e6:.0f} MB")
else:
    print("❌ No model to export")

In [ ]:
# Download the model
from google.colab import files
files.download("/content/garhwali_tts_model.zip")

## 📤 Deploy to HuggingFace

**Option 1 — Upload via CLI:**
```bash
pip install huggingface_hub
huggingface-cli login
huggingface-cli upload your-username/garhwali-tts ./garhwali_tts_export
```

**Option 2 — Create a Gradio Space:**
1. Go to [huggingface.co/new-space](https://huggingface.co/new-space)
2. Create a **Gradio** Space
3. Upload `model.pth`, `config.json` from the export
4. Use the `app.py` from your repo's `tts-training/huggingface/` folder

**Option 3 — Update PahadiTube server:**
1. Download `garhwali_tts_model.zip`
2. Extract into your server's TTS directory
3. Update the `/api/tts/synthesize` route to use the new model